# Phase 1 — Data Collection & Merging

Loads all 7 raw sources, standardises country names and years, and merges into one `master_dataset.csv`.

Output: `../data/processed/master_dataset.csv`

In [1]:
import pandas as pd
import numpy as np

RAW = '../data/raw/'
PROCESSED = '../data/processed/'

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 60)

## 1. Country reference list

We define the European countries we want to keep across all datasets.
Using ISO3 codes as the common key.

In [2]:
EUROPEAN_COUNTRIES = {
    'AUT': 'Austria',       'BEL': 'Belgium',       'BGR': 'Bulgaria',
    'HRV': 'Croatia',       'CYP': 'Cyprus',        'CZE': 'Czech Republic',
    'DNK': 'Denmark',       'EST': 'Estonia',       'FIN': 'Finland',
    'FRA': 'France',        'DEU': 'Germany',       'GRC': 'Greece',
    'HUN': 'Hungary',       'IRL': 'Ireland',       'ITA': 'Italy',
    'LVA': 'Latvia',        'LTU': 'Lithuania',     'LUX': 'Luxembourg',
    'MLT': 'Malta',         'NLD': 'Netherlands',   'NOR': 'Norway',
    'POL': 'Poland',        'PRT': 'Portugal',      'ROU': 'Romania',
    'SVK': 'Slovak Republic','SVN': 'Slovenia',     'ESP': 'Spain',
    'SWE': 'Sweden',        'CHE': 'Switzerland',   'GBR': 'United Kingdom',
    'ISL': 'Iceland',       'ALB': 'Albania',       'SRB': 'Serbia',
    'MKD': 'North Macedonia','MNE': 'Montenegro',   'BIH': 'Bosnia and Herzegovina'
}

# Region mapping for analysis
REGIONS = {
    'AUT': 'Western', 'BEL': 'Western', 'FRA': 'Western', 'DEU': 'Western',
    'IRL': 'Western', 'LUX': 'Western', 'NLD': 'Western', 'CHE': 'Western',
    'GBR': 'Western',
    'DNK': 'Northern', 'EST': 'Northern', 'FIN': 'Northern', 'ISL': 'Northern',
    'LVA': 'Northern', 'LTU': 'Northern', 'NOR': 'Northern', 'SWE': 'Northern',
    'CYP': 'Southern', 'GRC': 'Southern', 'ITA': 'Southern', 'MLT': 'Southern',
    'PRT': 'Southern', 'ESP': 'Southern',
    'ALB': 'Eastern', 'BIH': 'Eastern', 'BGR': 'Eastern', 'HRV': 'Eastern',
    'CZE': 'Eastern', 'HUN': 'Eastern', 'MKD': 'Eastern', 'MNE': 'Eastern',
    'POL': 'Eastern', 'ROU': 'Eastern', 'SRB': 'Eastern', 'SVK': 'Eastern',
    'SVN': 'Eastern'
}

print(f'Target: {len(EUROPEAN_COUNTRIES)} European countries')

Target: 36 European countries


## 2. WHO — Practising Nurses per 10,000

In [3]:
who_raw = pd.read_csv(RAW + 'HLTHRES_189_EN.csv', skiprows=20, low_memory=False)
print('Raw shape:', who_raw.shape)
who_raw.head()

Raw shape: (1353, 4)


,COUNTRY,COUNTRY_GRP,YEAR,VALUE
0,NaN,CARINFONET,1980.0,131.55
1,NaN,CARINFONET,1982.0,71.73
2,NaN,CARINFONET,1983.0,73.75
3,NaN,CARINFONET,1984.0,76.96
4,NaN,CARINFONET,1985.0,149.72


In [4]:
who = who_raw[['COUNTRY', 'YEAR', 'VALUE']].copy()
who.columns = ['iso3', 'year', 'nurses_per_10k']
who = who.dropna(subset=['iso3', 'year', 'nurses_per_10k'])
who['iso3'] = who['iso3'].str.strip()
who['year'] = who['year'].astype(int)
who['nurses_per_10k'] = pd.to_numeric(who['nurses_per_10k'], errors='coerce')

# Keep only European countries and years 2000+
who = who[who['iso3'].isin(EUROPEAN_COUNTRIES.keys()) & (who['year'] >= 2000)]
who = who.drop_duplicates(subset=['iso3', 'year'])

print('Cleaned shape:', who.shape)
print('Countries:', sorted(who['iso3'].unique()))
print('Years:', sorted(who['year'].unique()))
who.head(10)

Cleaned shape: (560, 3)
Countries: ['ALB', 'AUT', 'BEL', 'BGR', 'BIH', 'CHE', 'CYP', 'CZE', 'DEU', 'DNK', 'ESP', 'EST', 'FIN', 'GBR', 'GRC', 'HRV', 'HUN', 'IRL', 'ISL', 'ITA', 'LTU', 'LUX', 'LVA', 'MLT', 'NLD', 'NOR', 'POL', 'ROU', 'SRB', 'SVN', 'SWE']
Years: [2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021]


,iso3,year,nurses_per_10k
232,ALB,2012,44.53
233,ALB,2013,43.14
269,AUT,2000,55.46
270,AUT,2001,55.87
271,AUT,2002,56.79
272,AUT,2003,56.83
273,AUT,2004,59.28
274,AUT,2005,59.92
275,AUT,2006,61.45
276,AUT,2007,62.12


## 3. OECD — AMI & Stroke 30-day Mortality

In [5]:
oecd_raw = pd.read_csv(RAW + 'OECD.ELS.csv', low_memory=False)
print('Raw shape:', oecd_raw.shape)

# MORTAMII = AMI 30-day in-hospital mortality
# MORTISTI = Ischaemic stroke 30-day in-hospital mortality
oecd_outcomes = oecd_raw[oecd_raw['MEASURE'].isin(['MORTAMII', 'MORTISTI'])].copy()

# Keep total sex only (AGE is always Y_GE45 in this dataset — no age filter needed)
oecd_outcomes = oecd_outcomes[oecd_outcomes['SEX'] == '_T']

print('After filter:', oecd_outcomes.shape)
print('Measures:', oecd_outcomes['MEASURE'].unique())
print('SEX values:', oecd_outcomes['SEX'].unique())
print('AGE values:', oecd_outcomes['AGE'].unique())

Raw shape: (116363, 44)
After filter: (4044, 44)
Measures: ['MORTISTI' 'MORTAMII']
SEX values: ['_T']
AGE values: ['Y_GE45']


In [6]:
# Pivot to wide format: one row per country-year
oecd_outcomes['iso3'] = oecd_outcomes['REF_AREA']
oecd_outcomes['year'] = oecd_outcomes['TIME_PERIOD'].astype(int)
oecd_outcomes['value'] = pd.to_numeric(oecd_outcomes['OBS_VALUE'], errors='coerce')

oecd_wide = oecd_outcomes.groupby(['iso3', 'year', 'MEASURE'])['value'].mean().unstack('MEASURE').reset_index()
oecd_wide.columns.name = None

# Rename to our variable names
rename = {'MORTAMII': 'mortality_ami_30d', 'MORTISTI': 'mortality_stroke_30d'}
oecd_wide = oecd_wide.rename(columns=rename)

# Keep only European countries
oecd_wide = oecd_wide[oecd_wide['iso3'].isin(EUROPEAN_COUNTRIES.keys())]

print('OECD outcomes shape:', oecd_wide.shape)
print('Countries:', sorted(oecd_wide['iso3'].unique()))
oecd_wide.head(10)

OECD outcomes shape: (479, 4)
Countries: ['AUT', 'BEL', 'CHE', 'CZE', 'DEU', 'DNK', 'ESP', 'EST', 'FIN', 'FRA', 'GBR', 'HUN', 'IRL', 'ISL', 'ITA', 'LTU', 'LUX', 'LVA', 'MLT', 'NLD', 'NOR', 'POL', 'PRT', 'ROU', 'SVK', 'SVN', 'SWE']


,iso3,year,mortality_ami_30d,mortality_stroke_30d
24,AUT,2007,10.700000,8.300000
25,AUT,2008,9.700000,8.600000
26,AUT,2009,9.766667,8.100000
27,AUT,2010,9.433333,7.966667
28,AUT,2011,8.366667,7.800000
29,AUT,2012,7.833333,7.500000
30,AUT,2013,7.733333,7.566667
31,AUT,2014,7.300000,6.900000
32,AUT,2015,7.333333,7.033333
33,AUT,2016,6.500000,6.533333


## 4. OECD — Average Length of Stay

In [7]:
los_raw = pd.read_csv(RAW + 'OECD_ELS_HD_DSD_HEALTH_PROC.csv', low_memory=False)
print('Raw shape:', los_raw.shape)

# Filter to STAY measure, inpatient, all causes total
los = los_raw[
    (los_raw['MEASURE'] == 'STAY') &
    (los_raw['CARE_TYPE'].isin(['_T', 'T', 'Total'])) &
    (los_raw['FUNCTION'].isin(['_T', 'T', 'Total']))
].copy()

print('After filter:', los.shape)
los[['REF_AREA', 'Reference area', 'TIME_PERIOD', 'OBS_VALUE']].head(10)

Raw shape: (16242, 54)
After filter: (1729, 54)


,REF_AREA,Reference area,TIME_PERIOD,OBS_VALUE
0,AUS,Australia,1985,8.7
1,AUS,Australia,1987,8.2
2,AUS,Australia,1989,7.9
3,AUS,Australia,1991,7.3
4,AUS,Australia,1992,7.3
5,AUS,Australia,1993,7.1
6,AUS,Australia,1994,7.1
7,AUS,Australia,1995,6.7
8,AUS,Australia,1996,6.8
9,AUS,Australia,1997,6.7


In [8]:
los['iso3'] = los['REF_AREA']
los['year'] = los['TIME_PERIOD'].astype(int)
los['avg_length_of_stay'] = pd.to_numeric(los['OBS_VALUE'], errors='coerce')

los = los[['iso3', 'year', 'avg_length_of_stay']]
los = los[los['iso3'].isin(EUROPEAN_COUNTRIES.keys()) & (los['year'] >= 2000)]
los = los.dropna(subset=['avg_length_of_stay'])
los = los.groupby(['iso3', 'year'])['avg_length_of_stay'].mean().reset_index()

print('LOS shape:', los.shape)
print('Countries:', sorted(los['iso3'].unique()))
los.head(10)

LOS shape: (663, 3)
Countries: ['AUT', 'BEL', 'BGR', 'CHE', 'CZE', 'DEU', 'DNK', 'ESP', 'EST', 'FIN', 'FRA', 'GBR', 'GRC', 'HRV', 'HUN', 'IRL', 'ISL', 'ITA', 'LTU', 'LUX', 'LVA', 'NLD', 'NOR', 'POL', 'PRT', 'ROU', 'SVK', 'SVN', 'SWE']


,iso3,year,avg_length_of_stay
0,AUT,2000,9.8
1,AUT,2001,8.4
2,AUT,2002,8.4
3,AUT,2003,8.0
4,AUT,2004,8.4
5,AUT,2005,8.0
6,AUT,2006,7.9
7,AUT,2007,7.9
8,AUT,2008,7.9
9,AUT,2009,7.7


## 5. World Bank — Beds, Health Expenditure, GDP

In [9]:
def load_worldbank(filepath, varname):
    df = pd.read_csv(filepath, skiprows=4, low_memory=False)
    # Keep only country code + year columns
    year_cols = [c for c in df.columns if c.isdigit()]
    df = df[['Country Code'] + year_cols].copy()
    df = df[df['Country Code'].isin(EUROPEAN_COUNTRIES.keys())]
    # Melt to long format
    df = df.melt(id_vars='Country Code', var_name='year', value_name=varname)
    df.columns = ['iso3', 'year', varname]
    df['year'] = df['year'].astype(int)
    df[varname] = pd.to_numeric(df[varname], errors='coerce')
    df = df[df['year'] >= 2000].dropna(subset=[varname])
    return df

beds = load_worldbank(RAW + 'API_SH.MED.BEDS.ZS_DS2_en_csv_v2_115646.csv', 'beds_per_1000')
health_exp = load_worldbank(RAW + 'API_SH.XPD.CHEX.GD.ZS_DS2_en_csv_v2_115482.csv', 'health_exp_pct_gdp')
gdp = load_worldbank(RAW + 'API_NY.GDP.PCAP.CD_DS2_en_csv_v2_121663.csv', 'gdp_per_capita')

print('Beds:', beds.shape)
print('Health Exp:', health_exp.shape)
print('GDP:', gdp.shape)

Beds: (807, 3)
Health Exp: (870, 3)
GDP: (900, 3)


## 6. Eurostat — Cross-check Nurses (practising, not licensed)

In [10]:
estat_raw = pd.read_csv(RAW + 'eurostat_nurses.csv', low_memory=False)
print('Columns:', estat_raw.columns.tolist())
print('wstatus unique:', estat_raw['wstatus'].unique())
print('med_spec unique:', estat_raw['med_spec'].unique()[:10])
print('unit unique:', estat_raw['unit'].unique())

Columns: ['STRUCTURE', 'STRUCTURE_ID', 'STRUCTURE_NAME', 'freq', 'Time frequency', 'unit', 'Unit of measure', 'wstatus', 'Labour force and employment status', 'med_spec', 'Medical speciality', 'geo', 'Geopolitical entity (reporting)', 'TIME_PERIOD', 'Time', 'OBS_VALUE', 'Observation value', 'OBS_FLAG', 'Observation status (Flag) V2 structure', 'CONF_STATUS', 'Confidentiality status (flag)']
wstatus unique: ['LIC' 'PACT' 'PRACT']
med_spec unique: ['DENT' 'MWS' 'NRS' 'PHARM' 'PHYS' 'PER_CARE' 'PHYSIO']
unit unique: ['HAB_P' 'NR' 'P_HTHAB']


In [11]:
# Keep practising nurses, per 100k inhabitants
# Real codes: wstatus='PRACT', med_spec='NRS', unit='HAB_P'
estat = estat_raw[
    (estat_raw['wstatus'] == 'PRACT') &
    (estat_raw['med_spec'] == 'NRS') &
    (estat_raw['unit'] == 'HAB_P')
].copy()

print('After filter:', estat.shape)
estat[['geo', 'TIME_PERIOD', 'OBS_VALUE']].head(10)

After filter: (779, 21)


,geo,TIME_PERIOD,OBS_VALUE
8606,AT,1985,342.94
8607,AT,1986,329.13
8608,AT,1987,317.18
8609,AT,1988,317.91
8610,AT,1989,304.71
8611,AT,1990,295.32
8612,AT,1991,284.38
8613,AT,1992,269.53
8614,AT,1993,253.01
8615,AT,1994,234.85


In [12]:
# Note: Eurostat uses 2-letter ISO codes - map to ISO3
iso2_to_iso3 = {
    'AT':'AUT','BE':'BEL','BG':'BGR','HR':'HRV','CY':'CYP','CZ':'CZE',
    'DK':'DNK','EE':'EST','FI':'FIN','FR':'FRA','DE':'DEU','GR':'GRC',
    'HU':'HUN','IE':'IRL','IT':'ITA','LV':'LVA','LT':'LTU','LU':'LUX',
    'MT':'MLT','NL':'NLD','NO':'NOR','PL':'POL','PT':'PRT','RO':'ROU',
    'SK':'SVK','SI':'SVN','ES':'ESP','SE':'SWE','CH':'CHE','GB':'GBR',
    'IS':'ISL','AL':'ALB','RS':'SRB','MK':'MKD','ME':'MNE','BA':'BIH'
}

estat['iso3'] = estat['geo'].map(iso2_to_iso3)
estat['year'] = estat['TIME_PERIOD'].astype(int)
estat['nurses_eurostat_per_100k'] = pd.to_numeric(estat['OBS_VALUE'], errors='coerce')

estat = estat[['iso3', 'year', 'nurses_eurostat_per_100k']]
estat = estat.dropna(subset=['iso3', 'nurses_eurostat_per_100k'])
estat = estat[estat['year'] >= 2000]

print('Eurostat nurses shape:', estat.shape)
print('Countries:', sorted(estat['iso3'].unique()))

Eurostat nurses shape: (551, 3)
Countries: ['AUT', 'BEL', 'BGR', 'CHE', 'CYP', 'CZE', 'DEU', 'DNK', 'ESP', 'EST', 'FIN', 'FRA', 'HRV', 'HUN', 'IRL', 'ISL', 'ITA', 'LTU', 'LUX', 'LVA', 'MLT', 'MNE', 'NLD', 'NOR', 'POL', 'ROU', 'SRB', 'SVN', 'SWE']


## 7. Merge all datasets

WHO nurses is the primary staffing source. We merge everything on `iso3` + `year`.

In [13]:
# Build base: all country-year combos from 2000-2023
from itertools import product

all_iso3 = list(EUROPEAN_COUNTRIES.keys())
all_years = list(range(2000, 2024))
base = pd.DataFrame(list(product(all_iso3, all_years)), columns=['iso3', 'year'])
base['country'] = base['iso3'].map(EUROPEAN_COUNTRIES)
base['region'] = base['iso3'].map(REGIONS)
base['covid_period'] = base['year'].apply(
    lambda y: 'pre' if y <= 2019 else ('during' if y <= 2021 else 'post')
)

print('Base panel:', base.shape)

# Merge one by one
master = base.copy()
master = master.merge(who, on=['iso3', 'year'], how='left')
master = master.merge(oecd_wide, on=['iso3', 'year'], how='left')
master = master.merge(los, on=['iso3', 'year'], how='left')
master = master.merge(beds, on=['iso3', 'year'], how='left')
master = master.merge(health_exp, on=['iso3', 'year'], how='left')
master = master.merge(gdp, on=['iso3', 'year'], how='left')
master = master.merge(estat, on=['iso3', 'year'], how='left')

print('Master shape:', master.shape)
master.head(10)

Base panel: (864, 5)
Master shape: (864, 13)


,iso3,year,country,region,covid_period,nurses_per_10k,mortality_ami_30d,mortality_stroke_30d,avg_length_of_stay,beds_per_1000,health_exp_pct_gdp,gdp_per_capita,nurses_eurostat_per_100k
0,AUT,2000,Austria,Western,pre,55.46,NaN,NaN,9.8,7.95,9.389345,24487.297469,199.41
1,AUT,2001,Austria,Western,pre,55.87,NaN,NaN,8.4,7.85,9.452435,24430.495983,197.23
2,AUT,2002,Austria,Western,pre,56.79,NaN,NaN,8.4,7.81,9.591399,26334.862215,193.09
3,AUT,2003,Austria,Western,pre,56.83,NaN,NaN,8.0,7.73,9.698442,32110.115966,191.97
4,AUT,2004,Austria,Western,pre,59.28,NaN,NaN,8.4,7.73,9.815324,36614.250653,182.90
5,AUT,2005,Austria,Western,pre,59.92,NaN,NaN,8.0,7.69,9.717797,38157.370229,179.71
6,AUT,2006,Austria,Western,pre,61.45,NaN,NaN,7.9,7.66,9.640739,40382.207830,174.36
7,AUT,2007,Austria,Western,pre,62.12,10.700000,8.3,7.9,7.75,9.604826,46622.962291,171.93
8,AUT,2008,Austria,Western,pre,63.60,9.700000,8.6,7.9,7.69,9.834152,51581.398236,167.39
9,AUT,2009,Austria,Western,pre,64.75,9.766667,8.1,7.7,7.68,10.311529,47857.444664,164.00


In [14]:
# Engineer nurse change rate (YoY % change)
master = master.sort_values(['iso3', 'year'])
master['nurse_change_rate'] = master.groupby('iso3')['nurses_per_10k'].pct_change(fill_method=None) * 100

print('Final columns:', master.columns.tolist())
print('\nShape:', master.shape)

Final columns: ['iso3', 'year', 'country', 'region', 'covid_period', 'nurses_per_10k', 'mortality_ami_30d', 'mortality_stroke_30d', 'avg_length_of_stay', 'beds_per_1000', 'health_exp_pct_gdp', 'gdp_per_capita', 'nurses_eurostat_per_100k', 'nurse_change_rate']

Shape: (864, 14)


In [15]:
# Missing data summary
missing = (master.isnull().sum() / len(master) * 100).round(1)
print('Missing % per column:')
print(missing.to_string())

Missing % per column:
iso3                         0.0
year                         0.0
country                      0.0
region                       0.0
covid_period                 0.0
nurses_per_10k              35.2
mortality_ami_30d           45.3
mortality_stroke_30d        48.1
avg_length_of_stay          23.5
beds_per_1000                6.6
health_exp_pct_gdp           1.3
gdp_per_capita               0.0
nurses_eurostat_per_100k    36.5
nurse_change_rate           39.2


In [16]:
# Save
master.to_csv(PROCESSED + 'master_dataset.csv', index=False)
print(f'Saved master_dataset.csv — {master.shape[0]} rows x {master.shape[1]} columns')
master.describe().round(2)

Saved master_dataset.csv — 864 rows x 14 columns


,year,nurses_per_10k,mortality_ami_30d,mortality_stroke_30d,avg_length_of_stay,beds_per_1000,health_exp_pct_gdp,gdp_per_capita,nurses_eurostat_per_100k,nurse_change_rate
count,864.00,560.00,473.00,448.00,661.00,807.00,853.00,864.00,549.00,525.00
mean,2011.50,78.55,7.77,10.45,8.20,5.06,8.22,30837.62,283.29,1.48
std,6.93,35.08,2.82,4.49,1.59,1.77,1.75,25200.66,825.70,3.81
min,2000.00,26.91,2.47,3.47,4.90,1.90,4.21,974.73,61.41,-12.29
25%,2005.75,51.96,5.83,7.62,7.20,3.50,6.74,11617.73,105.49,0.05
50%,2011.50,68.98,7.40,9.52,8.10,4.86,8.25,23929.92,145.82,1.31
75%,2017.25,99.37,9.27,11.84,9.10,6.59,9.53,45487.10,205.05,2.42
max,2023.00,183.73,17.77,28.47,13.10,9.16,12.72,134965.82,15494.38,51.10
